In [1]:
import numpy as np, os

ROOT    = os.path.expanduser("~/things_eeg")
EEG_DIR = os.path.join(ROOT, "eeg_data/sub-01_63/sub-01__63_channels")
OUT_DIR = os.path.join(ROOT, "eeg_prepared")
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
def load_and_average(split):
    fname = "preprocessed_eeg_training.npy" if split == "train" else "preprocessed_eeg_test.npy"
    d = np.load(os.path.join(EEG_DIR, fname), allow_pickle=True).item()
    data     = d["preprocessed_eeg_data"]          # [koşul, tekrar, 64, 100]
    ch_names = list(d["ch_names"])
    times    = np.asarray(d["times"])
    # 1) stim kanalını at
    keep   = [i for i, c in enumerate(ch_names) if c != "stim"]
    data   = data[:, :, keep, :]                   # [koşul, tekrar, 63, 100]
    eeg_ch = [ch_names[i] for i in keep]
    # 2) tekrarları ortala (axis=1)
    data_avg = data.mean(axis=1).astype(np.float32)  # [koşul, 63, 100]
    return data_avg, eeg_ch, times

train_avg, ch, times = load_and_average("train")
test_avg,  _,  _     = load_and_average("test")

In [4]:
print("train_avg:", train_avg.shape)   # (16540, 63, 100)
print("test_avg :", test_avg.shape)    # (200, 63, 100)
print("kanal     :", len(ch), "| stim atıldı mı:", "stim" not in ch)
print("times     :", round(float(times.min()),3), "..", round(float(times.max()),3),
      "s | onset idx:", int(np.argmin(np.abs(times))))

np.save(os.path.join(OUT_DIR, "sub-01_train_avg.npy"), train_avg)
np.save(os.path.join(OUT_DIR, "sub-01_test_avg.npy"),  test_avg)
np.save(os.path.join(OUT_DIR, "eeg_channels.npy"), np.array(ch))
np.save(os.path.join(OUT_DIR, "eeg_times.npy"),    times)
print("saved ->", OUT_DIR)

train_avg: (16540, 63, 100)
test_avg : (200, 63, 100)
kanal     : 63 | stim atıldı mı: True
times     : -0.2 .. 0.79 s | onset idx: 20
saved -> /home/feyzanur_mbb/things_eeg/eeg_prepared


In [5]:
WINDOWS = {
    "baseline": (10, 20),   # -100..0 ms  (kontrol)
    "0_100":    (20, 30),
    "100_200":  (30, 40),
    "200_300":  (40, 50),
    "300_400":  (50, 60),
    "400_500":  (60, 70),
    "500_600":  (70, 80),
    "600_700":  (80, 90),
    "700_800":  (90, 100),
}

In [6]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "3")  # set before importing torch

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
ROOT     = os.path.expanduser("~/things_eeg")
EEG_DIR  = os.path.join(ROOT, "eeg_prepared")
FEAT_DIR = os.path.join(ROOT, "features")

WINDOWS = {
    "baseline": (10, 20), "0_100": (20, 30), "100_200": (30, 40),
    "200_300": (40, 50), "300_400": (50, 60), "400_500": (60, 70),
    "500_600": (70, 80), "600_700": (80, 90), "700_800": (90, 100),
}

class EEGDataset(Dataset):
    """Pair each image's EEG window with that image's target feature vector."""
    def __init__(self, eeg, targets, window):
        s, e = window
        self.eeg     = torch.from_numpy(eeg[:, :, s:e]).float()   # [N, 63, win]
        self.targets = torch.from_numpy(targets).float()          # [N, dim]
    def __len__(self):
        return len(self.eeg)
    def __getitem__(self, i):
        return self.eeg[i], self.targets[i]

class EEGDecoder(nn.Module):
    """2-layer MLP with a residual connection (same architecture as the original project)."""
    def __init__(self, input_size, output_size, hidden_size=1024):
        super().__init__()
        self.layer1   = nn.Linear(input_size, hidden_size)
        self.layer2   = nn.Linear(hidden_size, output_size)
        self.gelu     = nn.GELU()
        self.dropout  = nn.Dropout(0.1)
        self.residual = nn.Linear(input_size, output_size)
    def forward(self, x):
        x   = x.view(x.shape[0], -1)                  # flatten -> [N, 63*win]
        out = self.dropout(self.gelu(self.layer1(x)))
        out = self.layer2(out)
        return out + self.residual(x)

def info_nce(eeg_emb, tgt_emb, temperature=0.07):
    """Symmetric InfoNCE with in-batch negatives."""
    eeg    = F.normalize(eeg_emb, dim=-1)
    tgt    = F.normalize(tgt_emb, dim=-1)
    logits = eeg @ tgt.t() / temperature
    labels = torch.arange(len(eeg), device=eeg.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))

print("device:", DEVICE)

device: cuda


In [8]:
WINDOW_NAME = "100_200"
TARGET      = "RN50__attnpool"
SEED, EPOCHS = 0, 50
torch.manual_seed(SEED); np.random.seed(SEED)

train_eeg = np.load(os.path.join(EEG_DIR, "sub-01_train_avg.npy"))
test_eeg  = np.load(os.path.join(EEG_DIR, "sub-01_test_avg.npy"))
train_tgt = np.load(os.path.join(FEAT_DIR, f"{TARGET}__train.npy"))
test_tgt  = np.load(os.path.join(FEAT_DIR, f"{TARGET}__test.npy"))

window  = WINDOWS[WINDOW_NAME]
in_dim  = train_eeg.shape[1] * (window[1] - window[0])
out_dim = train_tgt.shape[1]

# carve a 10% validation split from training (for epoch selection, no test leakage)
n     = len(train_eeg)
perm  = np.random.permutation(n)
n_val = n // 10
val_idx, tr_idx = perm[:n_val], perm[n_val:]

train_dl = DataLoader(EEGDataset(train_eeg[tr_idx], train_tgt[tr_idx], window),
                      batch_size=256, shuffle=True, drop_last=True)

def to_batch(eeg, tgt, window):  # whole set as one tensor on device
    s, e = window
    return (torch.from_numpy(eeg[:, :, s:e]).float().to(DEVICE),
            torch.from_numpy(tgt).float().to(DEVICE))
val_eeg_t,  val_tgt_t  = to_batch(train_eeg[val_idx], train_tgt[val_idx], window)
test_eeg_t, test_tgt_t = to_batch(test_eeg, test_tgt, window)

model = EEGDecoder(in_dim, out_dim).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

@torch.no_grad()
def eval_loss(eeg_t, tgt_t):
    model.eval()
    return info_nce(model(eeg_t), tgt_t).item()

@torch.no_grad()
def test_top1():
    model.eval()
    emb  = model(test_eeg_t)
    sims = F.normalize(emb, dim=-1) @ F.normalize(test_tgt_t, dim=-1).t()
    return (sims.argmax(1) == torch.arange(len(emb), device=DEVICE)).float().mean().item()

best_val, best_test, best_epoch, best_top1 = float("inf"), None, 0, 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    for eeg, tgt in train_dl:
        eeg, tgt = eeg.to(DEVICE), tgt.to(DEVICE)
        opt.zero_grad(); info_nce(model(eeg), tgt).backward(); opt.step()
    sched.step()
    vl = eval_loss(val_eeg_t, val_tgt_t)
    if vl < best_val:
        best_val, best_test, best_epoch, best_top1 = vl, eval_loss(test_eeg_t, test_tgt_t), epoch, test_top1()
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:02d} | val_loss {vl:.3f} | test_loss {eval_loss(test_eeg_t, test_tgt_t):.3f}")

print(f"\nBEST (by val): epoch {best_epoch} | val {best_val:.3f} | test_loss {best_test:.3f} | top1 {best_top1*100:.1f}%")

epoch 01 | val_loss 6.854 | test_loss 4.332
epoch 10 | val_loss 6.881 | test_loss 3.909
epoch 20 | val_loss 7.251 | test_loss 4.161
epoch 30 | val_loss 7.410 | test_loss 4.251
epoch 40 | val_loss 7.472 | test_loss 4.275
epoch 50 | val_loss 7.485 | test_loss 4.270

BEST (by val): epoch 4 | val 6.649 | test_loss 3.954 | top1 14.5%
